# PHYS 3810: QFT, phase patterns, and bit order

**Goal:** connect a quantum circuit with the complete complex Fourier matrix, and explain why phase information can be invisible in one measurement basis but recovered by an inverse transform.

This notebook covers the first **20 minutes of session 2**: a 5-minute phase/bit refresher and 15 minutes on QFT. The later QPE lesson will complete the planned 50-minute core and 75-minute expansion. The follow-up exercises below are additional practice, not extra uncounted class segments.

**Before class:** complete [setup/preflight](00_setup_preflight.ipynb), select **Python (PHYS 3810)**, then restart the kernel and run all cells to verify the environment. **During class:** restart the kernel and clear outputs, discuss each prediction first, then execute the next relevant cell. The assertions are mathematical guardrails; focus discussion on what their results mean.

No IBM account, credentials, GPU, or internet after installation is needed. This notebook imports [the canonical QFT implementation](../../../src/phys3810/qft.py).


## 1. Phase and bit-order refresher · 5 minutes

An amplitude is complex; its squared magnitude is a probability. Changing a phase can leave that probability unchanged while affecting later interference.

We label basis states by integers and displayed bitstrings. **Qubit 0 is the rightmost and least-significant bit.** For the default `011`, the integer index is 3, and qubits 1 and 0 are set. Reversing those three bits gives `110`, index 6, so this example can expose a reversal mistake.

**Predict:** what information would be lost if we inspected only squared magnitudes of a complex amplitude vector?

The reusable implementation supports exact QFT on **1–6 qubits**, retains every controlled-phase rotation, and returns an unmeasured circuit without extra qubits. The default lesson uses three qubits.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

# Search enclosing folders; no Git metadata or package installation of the source is needed.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "qft.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the extracted course folder together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Operator, Statevector
from src.phys3810.qft import qft_circuit

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."
print("Python:", sys.version.split()[0], "| Qiskit:", version("qiskit"))
print("Interpreter:", sys.executable)

# Editable lesson parameters. Keep the label width equal to the register width.
num_qubits = 3
basis_label = "011"
assert len(basis_label) == num_qubits and set(basis_label) <= {"0", "1"}
basis_index = int(basis_label, 2)
size = 2**num_qubits
print(f"Selected basis state {basis_label}: integer {basis_index}, N={size}")


def fourier_matrix(n):
    """Independent mathematical expectation, not a circuit construction."""
    indices = np.arange(2**n)
    return np.exp(2j * np.pi * np.outer(indices, indices) / (2**n)) / np.sqrt(2**n)


## 2. From the matrix to a circuit · 5 minutes

For $N=2^n$, our **forward** sign convention is

$$F_{j,k}=\frac{1}{\sqrt N}\exp\!\left(+\frac{2\pi i j k}{N}\right).$$

The $k$th column is the output amplitude vector for input $|k\rangle$. The positive exponent matters: the inverse uses the complex-conjugate transpose, $F^\dagger$, with a negative exponent.

The circuit uses Hadamard and controlled-phase gates, followed by swaps that restore the displayed integer ordering. The complete matrix comparison below checks every basis input, including phases; `QFTGate` supplies a separate SDK reference check.

**Predict for the defaults:** what is the first amplitude of $F|3\rangle$? What angle does the next amplitude have? Do these values depend on the same aspects of the input?


In [ ]:
forward = qft_circuit(num_qubits)
expected_fourier = fourier_matrix(num_qubits)
np.testing.assert_allclose(Operator(forward).data, expected_fourier, rtol=0, atol=1e-12)
np.testing.assert_allclose(Operator(forward).data, Operator(QFTGate(num_qubits)).data, rtol=0, atol=1e-12)
assert forward.num_clbits == 0 and forward.num_ancillas == 0
print(f"PASSED: full {size} x {size} positive-exponent Fourier matrix and SDK reference")
print("Gate counts:", dict(forward.count_ops()))

figure = forward.draw("mpl", fold=24)
display(figure)
plt.close(figure)


## 3. A phase pattern can have uniform probabilities · 5 minutes

For a basis input $|k\rangle$, every transformed amplitude has magnitude $1/\sqrt N$. Therefore direct computational-basis measurement of $F|k\rangle$ is uniform for every $k$. That does **not** mean the output states are identical: their relative phases differ.

The upper figure panel shows real and imaginary amplitudes as the output index changes. The lower panel shows their squared magnitudes. Its equal-height bars cannot reveal the winding pattern above.

**Predict before running:** if we change the selected basis label from `011` to `001` while keeping three qubits, which panel changes? Would a counts-only test detect the sign of the Fourier exponent?


In [ ]:
basis = Statevector.from_label(basis_label)
transformed = basis.evolve(forward)
output_indices = np.arange(size)
expected_ramp = np.exp(2j * np.pi * basis_index * output_indices / size) / np.sqrt(size)
np.testing.assert_allclose(transformed.data, expected_ramp, rtol=0, atol=1e-12)
np.testing.assert_allclose(transformed.probabilities(), np.full(size, 1/size), rtol=0, atol=1e-12)
np.testing.assert_allclose(np.vdot(transformed.data, transformed.data), 1.0, rtol=0, atol=1e-12)

figure, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(output_indices, transformed.data.real, "o-", label="Real amplitude")
axes[0].plot(output_indices, transformed.data.imag, "s--", label="Imaginary amplitude")
axes[0].axhline(0, color="0.5", linewidth=0.7)
axes[0].set(ylabel="Amplitude component", title=f"Positive QFT of |{basis_label}> (k={basis_index}, N={size})")
axes[0].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
axes[0].grid(alpha=0.2)
axes[1].bar(output_indices, transformed.probabilities(), color="tab:green", width=0.65)
axes[1].set(xlabel="Output basis label j", ylabel="Probability", ylim=(0, 1.35/size))
axes[1].set_xticks(output_indices, [format(j, f"0{num_qubits}b") for j in output_indices], rotation=45)
axes[1].grid(axis="y", alpha=0.2)
figure.tight_layout()
display(figure)
plt.close(figure)
print(f"PASSED: phase ramp for {basis_label}; all {size} probabilities equal 1/{size}")


## 4. Recover a label from a known phase pattern · 5 minutes

The inverse must undo the **whole** forward transform: reverse gate order and conjugate the phases. Thus $F^\dagger F=I$.

We can also prepare this particular phase pattern independently. Apply a Hadamard to each qubit $q$, then a phase gate with angle $2\pi k2^q/N$. The resulting amplitude at index $j$ is $e^{2\pi i k j/N}/\sqrt N$. Inverse QFT converts that pattern to $|k\rangle$.

**Predict:** what displayed bitstring should be measured for the selected parameters? What would `110` instead of `011` imply for the default case?

We deliberately use a **known** integer $k$ to prepare the pattern. This is not QPE yet: there is no unknown eigenphase, eigenstate, or controlled sequence of powers. QPE will provide a different way to produce a phase pattern in a counting register.


In [ ]:
inverse = qft_circuit(num_qubits, inverse=True)
np.testing.assert_allclose(Operator(inverse).data, expected_fourier.conj().T, rtol=0, atol=1e-12)
recovered_basis = transformed.evolve(inverse)
np.testing.assert_allclose(recovered_basis.data, basis.data, rtol=0, atol=1e-12)

# Independent product-state preparation of the known positive phase gradient.
gradient = QuantumCircuit(num_qubits)
for qubit in range(num_qubits):
    gradient.h(qubit)
    gradient.p(2 * np.pi * basis_index * 2**qubit / size, qubit)
np.testing.assert_allclose(Statevector.from_instruction(gradient).data, expected_ramp, rtol=0, atol=1e-12)

decoded = gradient.compose(inverse)
expected_probabilities = np.zeros(size)
expected_probabilities[basis_index] = 1
np.testing.assert_allclose(Statevector.from_instruction(decoded).probabilities(), expected_probabilities, rtol=0, atol=1e-12)
shots = 256
measured = decoded.measure_all(inplace=False)
counts = StatevectorSampler(seed=2026).run([measured], shots=shots).result()[0].data.meas.get_counts()
assert counts == {basis_label: shots}
assert decoded.num_clbits == 0
print("Inverse-QFT sample counts:", counts)
print(f"PASSED: the known phase pattern decodes to {basis_label}; this example is not QPE")


### Core exit questions

Use these within the preceding five-minute inverse segment; they do not add a new timed section.

1. Why do equal measured probabilities not establish equal quantum states?
2. Why can a forward/inverse round trip alone miss a shared sign-convention error?
3. Which part of the known-gradient example supplied information that QPE must obtain differently?

<details>
<summary>Check your reasoning after discussing</summary>

Relative phases distinguish the states even when amplitude magnitudes agree. A pair of consistently reversed sign conventions can still undo each other; the independent complete Fourier matrix fixes the intended sign. The example already knows the integer used to set its preparation angles. QPE instead uses an eigenstate and controlled powers of a unitary to create the counting-register phase pattern.

</details>


## Follow-up practice: complex states

This practice is outside the scheduled 20-minute QFT introduction. All its code still runs during environment validation.

An operator acts on superpositions as well as basis states. Use a seeded, normalized complex vector to check $F|\psi\rangle$ and exact recovery by $F^\dagger$. Constructing a `Statevector` here is a classical simulator operation; it does not make arbitrary quantum state preparation free.


In [ ]:
rng = np.random.default_rng(2026)
input_data = rng.normal(size=size) + 1j * rng.normal(size=size)
input_data /= np.linalg.norm(input_data)
input_state = Statevector(input_data)
output_state = input_state.evolve(forward)
np.testing.assert_allclose(output_state.data, expected_fourier @ input_data, rtol=0, atol=1e-12)
np.testing.assert_allclose(np.vdot(output_state.data, output_state.data), 1.0, rtol=0, atol=1e-12)
np.testing.assert_allclose(output_state.evolve(inverse).data, input_data, rtol=0, atol=1e-12)
print("PASSED: complex-state matrix action, normalization, and inverse recovery")


## Follow-up practice: what changes when swaps are omitted?

Let $R$ reverse the bits of a basis index; $R=R^\dagger=R^{-1}$. With swaps, the forward circuit is $F$. Without them, it is **$RF$**, which reverses the output-index ordering.

The inverse is the adjoint of whichever forward circuit we selected:

- With swaps: $F^\dagger$.
- Without swaps: $(RF)^\dagger=F^\dagger R$.

The reversal is on the **input** side of the no-swaps inverse, not its output side. Reversing the entire gate sequence moves the swaps to the beginning when they are included.

**Predict for three qubits:** which indices are exchanged by $R$? In particular, where does `011` move? What must happen if we use the no-swaps forward circuit followed by its own inverse?


In [ ]:
reversed_indices = [int(format(j, f"0{num_qubits}b")[::-1], 2) for j in range(size)]
reversal = np.eye(size)[reversed_indices, :]
no_swaps_forward = qft_circuit(num_qubits, do_swaps=False)
no_swaps_inverse = qft_circuit(num_qubits, inverse=True, do_swaps=False)
np.testing.assert_allclose(Operator(no_swaps_forward).data, reversal @ expected_fourier, rtol=0, atol=1e-12)
np.testing.assert_allclose(Operator(no_swaps_inverse).data, expected_fourier.conj().T @ reversal, rtol=0, atol=1e-12)
np.testing.assert_allclose(
    input_state.evolve(no_swaps_forward).evolve(no_swaps_inverse).data,
    input_data, rtol=0, atol=1e-12,
)
print(f"Bit reversal maps {basis_label} to {basis_label[::-1]}")
print("PASSED: forward without swaps is R F; its inverse is F-dagger R")


## Further exercises and limits

- Keep three qubits and change `basis_label` to `001` or `110`. Predict the phase pattern and inverse sample label before rerunning. Expected indices, figures, and count labels are derived from your selected parameters.
- Change `num_qubits` and supply a matching-width binary label. Use 2–4 qubits for readable plots; the source and independent tests cover 1–6. Compare controlled-phase gate counts with $n(n-1)/2$.
- Explain why inspecting only output probabilities would not validate the Fourier sign, and why omitted swaps must be stated explicitly.

The logical circuit has a quadratic number of controlled-phase gates. Full matrix simulation and checking all its entries grow exponentially, and arbitrary input preparation and full amplitude readout have costs. These small local examples do not establish a general classical-FFT speedup or quantum advantage.

The [independent tests](../../../tests/test_qft.py) check complete matrices, inverse and swap conventions, complex-state recovery, asymmetric examples, invalid inputs, and V2 sampling. Run `python scripts/validate_classroom.py` from the repository root for all required checks. Source notebooks remain output-free; executed copies go under `artifacts/executed-classroom/`.

References: [IBM QFTGate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.QFTGate) specifies the positive-exponent transform; [IBM bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering) explains indices and displayed labels. See [the QFT contract](../QFT_PLAN.md) and [course plan](../COURSE_PLAN.md) for the subsequent QPE work. The combined QFT/QPE review precedes HHL/Shor implementation.
